In [1]:
# import dataset 
import pandas as pd 

# view results 
df = pd.read_csv('fraud_risk_dataset.csv')
df.head()

,transaction_id,customer_age,account_tenure_months,region,transaction_amount,merchant_category,payment_method,device_type,num_transactions_24h,international,previous_chargebacks,fraud
0,100000,65,68,South,130.13,Travel,Digital Wallet,Tablet,4,0,0,0
1,100001,22,4,South,28.27,Travel,Debit,Tablet,1,0,0,0
2,100002,43,53,South,304.83,Travel,Credit,Tablet,2,0,0,0
3,100003,72,69,West,39.87,Restaurants,Debit,Tablet,1,0,1,1
4,100004,21,8,South,125.73,Gas,Debit,Desktop,3,0,0,0


In [2]:
# Feature engineering (account maturity)

# Account Maturity x Pevious chargebacks 
df['tenure_x_chargebacks'] = (df['account_tenure_months'] * df['previous_chargebacks'])

# Account Maturity x velocity 
df['tenure_x_velocity'] = (df['account_tenure_months'] / df['num_transactions_24h'])

# Account Maturity x Transaction Amount 
df['tenure_x_velocity']  = (df['num_transactions_24h'] * df['account_tenure_months'])



In [3]:
# feature enginering (reputation)

# Reputation vs Region 
df['chargeback_x_region'] = (df['previous_chargebacks'] * df['region'])

# Reputation vs Device 
df['chargeback_x_device'] = (df['previous_chargebacks'] * df['device_type'])

# Reputation vs Customer age 
df['chargebacks_x_age'] = (df['previous_chargebacks'] * df['customer_age'])

# Reputations vs International 
df['chargebacks_x_international'] = (df['previous_chargebacks'] * df['international'])


In [4]:
# feature engineering (patterns and risk)

df['youth_risk'] = (df['customer_age'] < 30) & (df['num_transactions_24h'] >= 5 )

df['high_foreign_amount'] = (df['transaction_amount'] > 350 ) & (df['international'] == 1)


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


X = df.drop(columns = ['fraud'])
y = df['fraud']


#auto detect categorical vs numerical columns 
num_col = X.select_dtypes(include = ['number']).columns.to_list()
cat_col = X.select_dtypes(include = ['object']).columns.to_list()

# preprocesser
prep = ColumnTransformer(transformers = [
    ('num_col', StandardScaler(), num_col),
    ('cat_col', OneHotEncoder(handle_unknown= 'ignore'),
     cat_col)])


In [6]:
# Time to train the data

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.30,
                                                    stratify= y, random_state= 42)



In [7]:
pipe = Pipeline(steps= [
    ('preprocesser', prep),
    ('classifier', LogisticRegression(class_weight = 'balanced', 
                                      max_iter = 500))])


In [8]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('preprocesser',
                 ColumnTransformer(transformers=[('num_col', StandardScaler(),
                                                  ['transaction_id',
                                                   'customer_age',
                                                   'account_tenure_months',
                                                   'transaction_amount',
                                                   'num_transactions_24h',
                                                   'international',
                                                   'previous_chargebacks',
                                                   'tenure_x_chargebacks',
                                                   'tenure_x_velocity',
                                                   'chargebacks_x_age',
                                                   'chargebacks_x_international']),
                                                 ('cat_col',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['region',
                                                   'merchant_category',
                                                   'payment_method',
                                                   'device_type',
                                                   'chargeback_x_region',
                                                   'chargeback_x_device'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=500))])

In [9]:
y_preds = pipe.predict(X_test)

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# print results 
print('The evaluation report')
print(classification_report(y_test, y_preds))
print(confusion_matrix(y_test, y_preds))

The evaluation report
              precision    recall  f1-score   support

           0       0.96      0.76      0.85      1391
           1       0.17      0.61      0.26       109

    accuracy                           0.75      1500
   macro avg       0.56      0.69      0.56      1500
weighted avg       0.90      0.75      0.81      1500

[[1056  335]
 [  42   67]]


In [10]:
# Lets try using tuning threshold to catch fraud even if the probability is lower

In [11]:
pipe_proba = pipe.predict_proba(X_test)[:,1]
thresholds = [0.30, 0.40, 0.1, 0.05, 0.20]

for t in thresholds:
    preds = (pipe_proba >= t).astype(int)

#print results
print(f'- Threshold {t} - ')
print(classification_report(y_test, preds))


- Threshold 0.2 - 
              precision    recall  f1-score   support

           0       0.98      0.07      0.13      1391
           1       0.08      0.98      0.14       109

    accuracy                           0.14      1500
   macro avg       0.53      0.53      0.14      1500
weighted avg       0.91      0.14      0.13      1500



In [12]:
# Looking at the evaluated metrics, we should look to involve another model 
# A model that can optimze imbalanced datsets
# The baseline model currenlty is failing moving foward we will try an ensembel
# Let's bring in and test an a model that is robust to imbalance datsets

In [13]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline(steps= [
                ('prep', prep),
                ('model', RandomForestClassifier(
                    n_estimators= 300, # how many trees model will build 
                    max_depth = None, # How deep each tree can grow (model complexity)
                    min_samples_split= 10, # Minimun number of samples in each split
                    min_samples_leaf = 5, # Minimun number of leafs in each tree
                    class_weight= 'balanced', # balance terget variable 
                    n_jobs = -1, # Use all cpu for faster training 
                    random_state = 42, # Ensures reproducitbity
))])



rf.fit(X_train, y_train)



Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num_col', StandardScaler(),
                                                  ['transaction_id',
                                                   'customer_age',
                                                   'account_tenure_months',
                                                   'transaction_amount',
                                                   'num_transactions_24h',
                                                   'international',
                                                   'previous_chargebacks',
                                                   'tenure_x_chargebacks',
                                                   'tenure_x_velocity',
                                                   'chargebacks_x_age',
                                                   'chargebacks_x_international']),
                                                 ('cat_col',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['region',
                                                   'merchant_category',
                                                   'payment_method',
                                                   'device_type',
                                                   'chargeback_x_region',
                                                   'chargeback_x_device'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced',
                                        min_samples_leaf=5,
                                        min_samples_split=10, n_estimators=300,
                                        n_jobs=-1, random_state=42))])

In [14]:
# Let's view the classification report for the model and determine if we need threshold tunning or not

rf_pred = rf.predict(X_test) # Prediction 

print(f'classification_report{classification_report(y_test, rf_pred)}') #results

classification_report              precision    recall  f1-score   support

           0       0.94      0.95      0.95      1391
           1       0.25      0.20      0.22       109

    accuracy                           0.90      1500
   macro avg       0.59      0.58      0.58      1500
weighted avg       0.89      0.90      0.89      1500



In [15]:
# Let's view results with threshold tunning invovled 

rf_pred = rf.predict_proba(X_test)[:, 1] # Probability prediction (No binanry output)
thresholds = [0.15, 0.20, 0.25, 0.27, 0.30] # different thresholds 

for t in thresholds:
    proba = (rf_pred >= t).astype(int)
    
    print(f' Thresholds {t}')
    print(f'roc auc score {roc_auc_score(y_test, proba)}')
    print(f' Classification Report {classification_report(y_test, proba)}')

 Thresholds 0.15
roc auc score 0.6303134831386568
 Classification Report               precision    recall  f1-score   support

           0       0.97      0.43      0.60      1391
           1       0.10      0.83      0.18       109

    accuracy                           0.46      1500
   macro avg       0.54      0.63      0.39      1500
weighted avg       0.91      0.46      0.57      1500

 Thresholds 0.2
roc auc score 0.6941610220354968
 Classification Report               precision    recall  f1-score   support

           0       0.97      0.66      0.79      1391
           1       0.14      0.72      0.24       109

    accuracy                           0.67      1500
   macro avg       0.56      0.69      0.51      1500
weighted avg       0.91      0.67      0.75      1500

 Thresholds 0.25
roc auc score 0.7230723062413023
 Classification Report               precision    recall  f1-score   support

           0       0.97      0.78      0.86      1391
           1       

In [16]:
# Moving forward let's hyper tune the model utilzing the 0.28 threshold. This so far has produced the best results in eval. 

In [17]:
# Let's being modeling for RandomSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, roc_auc_score

param_dict = {
    'model__n_estimators': [100, 200, 300, 400, 500], # Number of trees (more trees = smoother predictions)
    'model__max_depth': [None, 3, 5, 6, 10,], # How deep each tree can grow (smaller depth -> fewer splits = less overfitting)
    'model__min_samples_leaf': [2, 4, 6, 8, 10], # Minimum number of samples before each split at a node (higher = more conservative)
    'model__min_samples_split': [2, 4, 5, 10], # Minimum number of samples in each split (forces generalization)
    'model__max_features': ['sqrt', 'log2', 0.5] # adjusting the allowed features per each split (forces randomness = better generalization)
}
rf_tuner = RandomizedSearchCV(
    estimator = rf, # Model
    param_distributions = param_dict, # the knobs model is allowed to adjust
    n_iter = 20, # Amount of iterations model is allowed to perform
    scoring = 'roc_auc', # Optimizing for evalutation metric 
    cv = 5, # Split data in 5 cross vals
    verbose = 1, # Demonstrate what you are doing 
    n_jobs = -1, # Use all of the cpu's for faster training 
    random_state = 42 # ensure reproducibiltiy
)
    
rf_tuner.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


C:\Users\dmiracju\AppData\Roaming\Python\Python313\site-packages\numpy\ma\core.py:2896: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('prep',
                                              ColumnTransformer(transformers=[('num_col',
                                                                               StandardScaler(),
                                                                               ['transaction_id',
                                                                                'customer_age',
                                                                                'account_tenure_months',
                                                                                'transaction_amount',
                                                                                'num_transactions_24h',
                                                                                'international',
                                                                                'previous_chargebacks',
                                                                                'tenure_x_chargebacks',
                                                                                'tenure_x_velocity',
                                                                                'chargebacks_x_age',
                                                                                'chargebacks_x_international'...
                                                                     min_samples_split=10,
                                                                     n_estimators=300,
                                                                     n_jobs=-1,
                                                                     random_state=42))]),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'model__max_depth': [None, 3, 5, 6, 10],
                                        'model__max_features': ['sqrt', 'log2',
                                                                0.5],
                                        'model__min_samples_leaf': [2, 4, 6, 8,
                                                                    10],
                                        'model__min_samples_split': [2, 4, 5,
                                                                     10],
                                        'model__n_estimators': [100, 200, 300,
                                                                400, 500]},
                   random_state=42, scoring='roc_auc', verbose=1)

In [18]:
# Evaluate the best param results 
best_score = rf_tuner.best_score_
best_param = rf_tuner.best_params_ 
best_estimator = rf_tuner.best_estimator_
# Print
print(f'-- Best score --- {best_score}')
print(f'-- Best Parameters --- {best_param}')
print(f'--- Best Estimators --- {best_estimator}')

-- Best score --- 0.7291544828451707
-- Best Parameters --- {'model__n_estimators': 400, 'model__min_samples_split': 5, 'model__min_samples_leaf': 4, 'model__max_features': 'log2', 'model__max_depth': 3}
--- Best Estimators --- Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num_col', StandardScaler(),
                                                  ['transaction_id',
                                                   'customer_age',
                                                   'account_tenure_months',
                                                   'transaction_amount',
                                                   'num_transactions_24h',
                                                   'international',
                                                   'previous_chargebacks',
                                                   'tenure_x_chargebacks',
                                                   'tenure_x_velocity',
                  

In [19]:
y_prob = best_estimator.predict_proba(X_test)[:,1]
print('score', roc_auc_score(y_test, y_prob)) 


score 0.7255093359011734


In [20]:
# The best iteration of parameters optimized on ROC AUC score at 72%. 
# These results show that the model can tell a difference between the two classes. 

In [38]:
# Let's find the best probablity cuttoff 
from sklearn.metrics import precision_recall_curve

# for each threshold calculate precision, recall and store it as an array 
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

# Compute f1_score for each threshold 
f1_score = 2 * (precision * recall) / (precision + recall + 1e-10)

# find index of best f1
best_idx = np.argmax(f1_score[:-1])

#Extract the best threshold 
best_thresh = thresholds[best_idx]

# extrac metrics at best thresholds 
best_recall = recall[best_idx]
best_precision = precision[best_idx]
best_f1 = f1_score[best_idx]

In [39]:
# Print best results 
print('Best F1 threshold RF', best_thresh)
print('Best f1', best_f1)
print('precision @ best f1', best_precision)
print('Recall @ best f1', best_recall)

Best F1 threshold RF 0.513860871800065
Best f1 0.3307888040311611
precision @ best f1 0.22887323943661972
Recall @ best f1 0.5963302752293578


In [43]:
# Lets evaluate the probability cutt off using the best threshold 
y_pred = (y_prob >= best_thresh).astype(int) # Turn output into integer

print(f'classification_report {classification_report(y_test, y_pred)}')
print(f'confusion_matrix {confusion_matrix(y_test, y_pred)}')
print(f'roc auc {roc_auc_score(y_test, y_pred)}')

classification_report               precision    recall  f1-score   support

           0       0.96      0.84      0.90      1391
           1       0.23      0.60      0.33       109

    accuracy                           0.82      1500
   macro avg       0.60      0.72      0.61      1500
weighted avg       0.91      0.82      0.86      1500

confusion_matrix [[1172  219]
 [  44   65]]
roc auc 0.7194447925391937


In [66]:
# Feature importance 

rf_final = best_estimator.named_steps['model'] # grab feature names from preprocessing

importances = rf_final.feature_importances_ # return frature importance
feature_names = best_estimator.feature_names_in_ # Deploy features 
fI = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(ascending=False)

ValueError: All arrays must be of the same length

In [ ]:
# We are having issues with aligining the arrays. Please re work on feature importances 

array([3.98376805e-02, 2.84535259e-02, 3.13956952e-02, 1.65208370e-01,
       3.17090686e-02, 1.58297523e-01, 8.55741712e-02, 4.63902200e-02,
       2.88327806e-02, 7.12190931e-02, 3.44797491e-02, 1.61997859e-02,
       6.36292607e-03, 1.98501488e-03, 1.03896340e-02, 2.28596522e-03,
       2.86934807e-03, 2.62612116e-03, 3.30875704e-03, 2.78314148e-03,
       2.37137015e-03, 1.26625661e-03, 2.58592892e-03, 4.62802456e-03,
       3.89415189e-03, 3.77517290e-03, 2.24349719e-03, 5.02502361e-03,
       2.34324967e-03, 3.05450081e-03, 3.97176993e-03, 1.23115251e-02,
       0.00000000e+00, 0.00000000e+00, 2.43869839e-03, 2.87703805e-03,
       3.30894637e-03, 1.29354146e-02, 3.88326165e-02, 1.99323939e-03,
       0.00000000e+00, 9.56898590e-03, 1.31166218e-02, 3.29946794e-04,
       0.00000000e+00, 3.43978020e-03, 8.93639912e-03, 4.27916557e-02,
       5.60892892e-03, 7.72514315e-03, 5.01488788e-03, 6.38356072e-05,
       0.00000000e+00, 3.72057389e-03, 1.96182453e-02, 0.00000000e+00,
      